[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/04.%20The%20FCFS%20Berth%20Scheduling%20Problem/P4-Tier-1_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 4. The FCFS Berth Scheduling Problem

## Tier 1: A Network-Flow Formulation

### Purpose and inputs

We assign each arriving vessel to one compatible berth and choose its service start time. The objective is to minimize the sum of vessel waiting times. The inputs are vessel arrival times, processing times, optional vessel dimensions, berth capacities, a planning horizon, and a time step. The outputs are a feasible schedule, total and average waiting time, and the schedule makespan.

### Scope and modeling assumptions

- Vessels are considered in nondecreasing order of arrival time. Vessels with equal arrival times are ordered by vessel ID.
- FCFS means that service start times are nondecreasing in this arrival order. Two vessels may start at the same time on different berths; a later vessel cannot start before an earlier vessel.
- A berth serves at most one vessel at a time. A vessel occupies its assigned berth continuously for its processing time.
- Processing times and arrival times are deterministic. Setup times are zero, or are already included in processing times.
- Vessel dimensions are optional because they are not specified in the concrete example. When supplied, vessel length and draft must not exceed the corresponding berth capacities.
- Start times, arrivals, processing times, and the planning horizon must align with the selected time step. This example uses a 0.5-hour step.
- The model does not include tides, weather disruptions, berth-dependent processing rates, priorities, or uncertainty.

### From scheduling decisions to a network

A state records the next available time at every berth and the most recent vessel's start time. From a state at vessel position $i$, an arc represents assigning vessel $i$ to one compatible berth at its earliest feasible start time. The arc's cost is that vessel's waiting time. Reaching the sink after the last vessel corresponds to a complete schedule.

This is a layered, directed acyclic state-space network. A unit of flow follows one source-to-sink path, and each path encodes one FCFS-feasible schedule. The shortest path therefore gives a minimum-total-wait schedule in the discretized model. The code solves this network by dynamic programming over its layers; it does not invoke an external linear-programming solver.

### Mathematical model

Let vessels be indexed by $i=1,\ldots,n$ in FCFS order and berths by $b\in B$. Let $a_i$ be vessel $i$'s arrival time, $p_i$ its processing time, and $s_i$ its service start time. Let $A_b$ denote the time berth $b$ becomes available. A feasible transition for vessel $i$ on berth $b$ uses

$$
s_i = \max\{a_i, A_b, s_{i-1}\},
\qquad
A_b' = s_i+p_i,
\qquad
w_i=s_i-a_i.
$$

Here $w_i$ is waiting time; the transition is available only if the vessel fits berth $b$ and $s_i+p_i$ does not exceed the planning horizon. The other berth-availability times remain unchanged. The maximum enforces arrival feasibility, berth non-overlap, and nondecreasing FCFS start times. For the first vessel, take $s_0=0$; all arrival times are nonnegative.

The objective is

$$
\min \sum_{i=1}^{n} w_i.
$$

Equivalently, let $F_i(q)$ be the minimum accumulated waiting cost after scheduling the first $i$ vessels and reaching state $q$. The initial state has every berth available at time zero and last start time zero, with $F_0(q_0)=0$. For each feasible transition from state $q$ to state $q'$ with waiting cost $w_i$, the recurrence is

$$
F_i(q') = \min_{q\to q'}\left\{F_{i-1}(q)+w_i\right\}.
$$

The answer is the minimum cost of a complete path to the sink. In the code, the state is a tuple of berth-availability ticks and the previous start-time tick; transition costs are stored on `NetworkFlowArc` objects. The initialization and layer-by-layer relaxation in `solve_network_flow` implement the base case and recurrence. The number of reachable states can grow rapidly with the number of berths and time ticks, so this exact state-space network is intended for small teaching examples rather than unrestricted large instances.

### Concrete example

The example has two berths and four vessels:

| Vessel | Arrival (hours) | Processing time (hours) |
|:--|--:|--:|
| 1 | 0 | 3 |
| 2 | 1 | 2 |
| 3 | 2 | 4 |
| 4 | 3 | 1 |

The berth capacities are 400 m and 16 m for Berth 0, and 350 m and 14 m for Berth 1. Since the example gives no vessel dimensions, both berths are considered compatible with every vessel in this instance. A schedule with starts at 0, 1, 3, and 3 hours, respectively, is feasible under the stated FCFS rule; its total waiting time is the sum of each start time minus its arrival time.

### Interpreting the results

A lower total waiting time means vessels spend less time waiting after arrival. The makespan is the latest vessel completion time measured from time zero; it is reported as a secondary metric and is not the objective optimized by this model. Berth utilization is the fraction of that makespan occupied by processing. If several schedules have the same minimum waiting cost, the network can have multiple optimal paths; the code returns one of them.

In [ ]:
# These packages support the data structures, network representation, and visual analysis.
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Packages imported successfully for the network-flow formulation.")

In [ ]:
@dataclass
class Vessel:
    """A vessel's arrival, processing requirement, and optional dimensions."""

    id: int
    arrival_time: float
    processing_time: float
    max_length: Optional[float] = None
    max_draft: Optional[float] = None


@dataclass
class Berth:
    """A berth's maximum accommodated vessel length and draft."""

    id: int
    max_length: float
    max_draft: float


@dataclass
class NetworkFlowNode:
    """A source, state, or sink node in the layered scheduling network."""

    id: str
    node_type: str
    layer: Optional[int] = None
    state: Optional[Tuple[Tuple[int, ...], int]] = None


@dataclass
class NetworkFlowArc:
    """A capacity-one network arc, optionally representing a vessel assignment."""

    from_node: str
    to_node: str
    capacity: float
    cost: float
    vessel_id: Optional[int] = None
    berth_id: Optional[int] = None
    start_time: Optional[float] = None
    end_time: Optional[float] = None


print("Scheduling data structures are ready.")

In [ ]:
class NetworkFlowFormulation:
    """Exact shortest-path formulation for discretized FCFS berth scheduling."""

    def __init__(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        planning_horizon: float = 24.0,
        time_step: float = 1.0,
    ):
        if not vessels:
            raise ValueError("At least one vessel is required.")
        if not berths:
            raise ValueError("At least one berth is required.")
        if len({vessel.id for vessel in vessels}) != len(vessels):
            raise ValueError("Vessel IDs must be unique.")
        if len({berth.id for berth in berths}) != len(berths):
            raise ValueError("Berth IDs must be unique.")
        if time_step <= 0 or planning_horizon <= 0:
            raise ValueError("The time step and planning horizon must be positive.")

        horizon_ticks = planning_horizon / time_step
        if not math.isclose(horizon_ticks, round(horizon_ticks), abs_tol=1e-9):
            raise ValueError("The planning horizon must be an integer multiple of the time step.")

        self.vessels = vessels
        self.berths = berths
        self.planning_horizon = float(planning_horizon)
        self.time_step = float(time_step)
        self.time_periods = int(round(horizon_ticks))
        self.vessels_sorted = sorted(vessels, key=lambda vessel: (vessel.arrival_time, vessel.id))

        for vessel in self.vessels_sorted:
            if vessel.arrival_time < 0 or vessel.processing_time <= 0:
                raise ValueError("Arrival times must be nonnegative and processing times positive.")
            self._to_tick(vessel.arrival_time, "arrival time")
            self._to_tick(vessel.processing_time, "processing time")

        self.nodes: Dict[str, NetworkFlowNode] = {}
        self.arcs: List[NetworkFlowArc] = []
        self.transition_arcs: List[List[NetworkFlowArc]] = []
        self.node_counter = 0
        self.source_id: Optional[str] = None
        self.sink_id: Optional[str] = None
        self.solution: Optional[Dict] = None
        self.schedule: Optional[List[Dict]] = None

    def _to_tick(self, time_value: float, description: str) -> int:
        """Map a model time to an integer tick only when it lies on the time grid."""
        tick_value = time_value / self.time_step
        nearest_tick = round(tick_value)
        if not math.isclose(tick_value, nearest_tick, abs_tol=1e-9):
            raise ValueError(f"Each {description} must align with the selected time step.")
        return int(nearest_tick)

    def _add_node(
        self,
        node_type: str,
        layer: Optional[int] = None,
        state: Optional[Tuple[Tuple[int, ...], int]] = None,
    ) -> str:
        """Create a uniquely named node so each network state has one identity."""
        node_id = f"{node_type}_{self.node_counter}"
        self.node_counter += 1
        self.nodes[node_id] = NetworkFlowNode(
            id=node_id,
            node_type=node_type,
            layer=layer,
            state=state,
        )
        return node_id

    def _is_compatible(self, vessel: Vessel, berth: Berth) -> bool:
        """Check physical limits when the vessel's corresponding dimension is known."""
        length_fits = vessel.max_length is None or vessel.max_length <= berth.max_length
        draft_fits = vessel.max_draft is None or vessel.max_draft <= berth.max_draft
        return length_fits and draft_fits

    def _satisfies_fcfs(
        self,
        vessel: Vessel,
        berth: Berth,
        start_time: float,
        previous_start_time: float = 0.0,
    ) -> bool:
        """Check arrival feasibility and the nondecreasing FCFS start-time rule."""
        return start_time >= vessel.arrival_time and start_time >= previous_start_time

    def create_network_structure(self) -> None:
        """Build all reachable state transitions in the layered time-space network."""
        self.nodes = {}
        self.arcs = []
        self.transition_arcs = [[] for _ in self.vessels_sorted]
        self.node_counter = 0

        self.source_id = self._add_node("source", layer=0)
        initial_state = (tuple(0 for _ in self.berths), 0)
        state_nodes: List[Dict[Tuple[Tuple[int, ...], int], str]] = [
            {} for _ in range(len(self.vessels_sorted) + 1)
        ]
        state_nodes[0][initial_state] = self.source_id

        for vessel_index, vessel in enumerate(self.vessels_sorted):
            arrival_tick = self._to_tick(vessel.arrival_time, "arrival time")
            processing_ticks = self._to_tick(vessel.processing_time, "processing time")

            for state, from_node in list(state_nodes[vessel_index].items()):
                berth_available_ticks, previous_start_tick = state

                for berth_index, berth in enumerate(self.berths):
                    if not self._is_compatible(vessel, berth):
                        continue

                    start_tick = max(
                        arrival_tick,
                        berth_available_ticks[berth_index],
                        previous_start_tick,
                    )
                    end_tick = start_tick + processing_ticks
                    if end_tick > self.time_periods:
                        continue

                    start_time = start_tick * self.time_step
                    previous_start_time = previous_start_tick * self.time_step
                    if not self._satisfies_fcfs(
                        vessel, berth, start_time, previous_start_time
                    ):
                        continue

                    next_availability = list(berth_available_ticks)
                    next_availability[berth_index] = end_tick
                    next_state = (tuple(next_availability), start_tick)

                    if next_state not in state_nodes[vessel_index + 1]:
                        state_nodes[vessel_index + 1][next_state] = self._add_node(
                            "state",
                            layer=vessel_index + 1,
                            state=next_state,
                        )

                    waiting_time = start_time - vessel.arrival_time
                    arc = NetworkFlowArc(
                        from_node=from_node,
                        to_node=state_nodes[vessel_index + 1][next_state],
                        capacity=1.0,
                        cost=waiting_time,
                        vessel_id=vessel.id,
                        berth_id=berth.id,
                        start_time=start_time,
                        end_time=end_tick * self.time_step,
                    )
                    self.arcs.append(arc)
                    self.transition_arcs[vessel_index].append(arc)

        self.sink_id = self._add_node("sink", layer=len(self.vessels_sorted) + 1)
        for final_node in state_nodes[-1].values():
            self.arcs.append(
                NetworkFlowArc(
                    from_node=final_node,
                    to_node=self.sink_id,
                    capacity=1.0,
                    cost=0.0,
                )
            )

        print(
            f"Network created: {len(self.nodes)} nodes, "
            f"{len(self.arcs)} arcs, {len(self.vessels_sorted)} vessel layers."
        )

    def solve_network_flow(self) -> Dict:
        """Find a minimum-cost source-to-sink path by layered dynamic programming."""
        if self.source_id is None or self.sink_id is None:
            raise RuntimeError("Create the network structure before solving it.")

        distances = {self.source_id: 0.0}
        predecessor: Dict[str, NetworkFlowArc] = {}

        for layer_arcs in self.transition_arcs:
            for arc in layer_arcs:
                if arc.from_node not in distances:
                    continue
                candidate_cost = distances[arc.from_node] + arc.cost
                current_cost = distances.get(arc.to_node, math.inf)
                if candidate_cost < current_cost and not math.isclose(
                    candidate_cost, current_cost, rel_tol=1e-12, abs_tol=1e-12
                ):
                    distances[arc.to_node] = candidate_cost
                    predecessor[arc.to_node] = arc

        sink_arcs = [arc for arc in self.arcs if arc.to_node == self.sink_id]
        for arc in sink_arcs:
            if arc.from_node in distances:
                candidate_cost = distances[arc.from_node]
                current_cost = distances.get(self.sink_id, math.inf)
                if candidate_cost < current_cost and not math.isclose(
                    candidate_cost, current_cost, rel_tol=1e-12, abs_tol=1e-12
                ):
                    distances[self.sink_id] = candidate_cost
                    predecessor[self.sink_id] = arc

        if self.sink_id not in predecessor:
            raise ValueError(
                "No complete feasible schedule exists within the planning horizon."
            )

        path_arcs = []
        current_node = self.sink_id
        while current_node != self.source_id:
            arc = predecessor[current_node]
            path_arcs.append(arc)
            current_node = arc.from_node
        path_arcs.reverse()

        schedule = [
            {
                "vessel_id": arc.vessel_id,
                "berth_id": arc.berth_id,
                "start_time": arc.start_time,
                "end_time": arc.end_time,
                "waiting_time": arc.cost,
                "processing_time": arc.end_time - arc.start_time,
            }
            for arc in path_arcs
            if arc.vessel_id is not None
        ]

        if len(schedule) != len(self.vessels_sorted):
            raise AssertionError("The recovered path does not schedule every vessel.")

        self.schedule = schedule
        self.solution = {
            "schedule": schedule,
            "total_waiting_time": self._calculate_total_waiting_time(schedule),
            "makespan": self._calculate_makespan(schedule),
        }

        if not math.isclose(
            self.solution["total_waiting_time"],
            distances[self.sink_id],
            rel_tol=1e-9,
            abs_tol=1e-9,
        ):
            raise AssertionError("Path cost and reported waiting time do not agree.")

        return self.solution

    def _calculate_total_waiting_time(self, schedule: List[Dict]) -> float:
        """Sum the waiting-time cost of all vessel-assignment arcs."""
        return sum(assignment["waiting_time"] for assignment in schedule)

    def _calculate_makespan(self, schedule: List[Dict]) -> float:
        """Return the latest completion time, or zero for an empty schedule."""
        if not schedule:
            return 0.0
        return max(assignment["end_time"] for assignment in schedule)

    def visualize_schedule(self) -> None:
        """Plot berth occupancy and vessel waiting times with labeled axes."""
        if not self.schedule:
            print("No schedule is available to visualize.")
            return

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        vessel_colors = plt.cm.Set3(np.linspace(0, 1, len(self.vessels)))
        vessel_color = {
            vessel.id: vessel_colors[index]
            for index, vessel in enumerate(self.vessels)
        }

        for assignment in self.schedule:
            ax1.barh(
                assignment["berth_id"],
                assignment["processing_time"],
                left=assignment["start_time"],
                color=vessel_color[assignment["vessel_id"]],
                alpha=0.85,
                edgecolor="black",
                linewidth=1,
            )
            ax1.text(
                assignment["start_time"] + assignment["processing_time"] / 2,
                assignment["berth_id"],
                f"V{assignment['vessel_id']}",
                ha="center",
                va="center",
                fontweight="bold",
            )

        ax1.set_xlabel("Time (hours)")
        ax1.set_ylabel("Berth ID")
        ax1.set_yticks([berth.id for berth in self.berths])
        ax1.set_title("FCFS Berth Schedule")
        ax1.grid(True, alpha=0.3)
        ax1.set_xlim(0, max(self._calculate_makespan(self.schedule), 1))

        vessel_ids = [assignment["vessel_id"] for assignment in self.schedule]
        waiting_times = [assignment["waiting_time"] for assignment in self.schedule]
        bars = ax2.bar(
            vessel_ids,
            waiting_times,
            color="lightcoral",
            alpha=0.85,
            edgecolor="black",
        )
        for bar, waiting_time in zip(bars, waiting_times):
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{waiting_time:.1f}",
                ha="center",
                va="bottom",
            )

        ax2.set_xlabel("Vessel ID")
        ax2.set_ylabel("Waiting time (hours)")
        ax2.set_title("Waiting Time by Vessel")
        ax2.set_xticks(vessel_ids)
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    def print_solution_summary(self) -> None:
        """Print the schedule and check completeness and FCFS start order."""
        if not self.solution or not self.schedule:
            print("No solution is available.")
            return

        average_wait = self.solution["total_waiting_time"] / len(self.vessels)
        print("\n" + "=" * 64)
        print("NETWORK-FLOW SOLUTION SUMMARY")
        print("=" * 64)
        print(f"Vessels: {len(self.vessels)}")
        print(f"Berths: {len(self.berths)}")
        print(f"Planning horizon: {self.planning_horizon:.1f} hours")
        print(f"Time step: {self.time_step:.2f} hours")
        print(f"Total waiting time: {self.solution['total_waiting_time']:.1f} hours")
        print(f"Average waiting time: {average_wait:.2f} hours per vessel")
        print(f"Makespan: {self.solution['makespan']:.1f} hours")
        print("\nSchedule:")
        print(f"{'Vessel':<10}{'Berth':<10}{'Start':<10}{'End':<10}{'Wait':<10}{'Process':<10}")
        print("-" * 60)
        for assignment in self.schedule:
            print(
                f"V{assignment['vessel_id']:<9}"
                f"{assignment['berth_id']:<10}"
                f"{assignment['start_time']:<10.1f}"
                f"{assignment['end_time']:<10.1f}"
                f"{assignment['waiting_time']:<10.1f}"
                f"{assignment['processing_time']:<10.1f}"
            )

        ordered_assignments = sorted(
            self.schedule,
            key=lambda assignment: next(
                index
                for index, vessel in enumerate(self.vessels_sorted)
                if vessel.id == assignment["vessel_id"]
            ),
        )
        fcfs_satisfied = all(
            ordered_assignments[index]["start_time"]
            <= ordered_assignments[index + 1]["start_time"]
            for index in range(len(ordered_assignments) - 1)
        )
        print(f"\nEvery vessel scheduled: {len(self.schedule) == len(self.vessels)}")
        print(f"FCFS start-order constraint satisfied: {fcfs_satisfied}")


print("The network-flow formulation is defined.")

In [ ]:
# The example specifies arrival and processing times, but not vessel dimensions.
print("Creating the concrete berth-scheduling example.")

vessels = [
    Vessel(id=1, arrival_time=0.0, processing_time=3.0),
    Vessel(id=2, arrival_time=1.0, processing_time=2.0),
    Vessel(id=3, arrival_time=2.0, processing_time=4.0),
    Vessel(id=4, arrival_time=3.0, processing_time=1.0),
]

berths = [
    Berth(id=0, max_length=400, max_draft=16),
    Berth(id=1, max_length=350, max_draft=14),
]

print(f"Created {len(vessels)} vessels and {len(berths)} berths.")
print("\nVessel details:")
for vessel in vessels:
    print(
        f"Vessel {vessel.id}: arrival={vessel.arrival_time:.1f} h, "
        f"processing={vessel.processing_time:.1f} h, dimensions unspecified"
    )

print("\nBerth capacities:")
for berth in berths:
    print(
        f"Berth {berth.id}: maximum length={berth.max_length} m, "
        f"maximum draft={berth.max_draft} m"
    )

In [ ]:
# Construct the layered network, then find its minimum-cost complete path.
fcfs_formulation = NetworkFlowFormulation(
    vessels=vessels,
    berths=berths,
    planning_horizon=24.0,
    time_step=0.5,
)

fcfs_formulation.create_network_structure()
solution = fcfs_formulation.solve_network_flow()

print("A complete minimum-cost schedule was found.")

In [ ]:
fcfs_formulation.print_solution_summary()

# Check the example schedule and objective using tolerance-aware comparisons.
expected_schedule = [
    {"vessel_id": 1, "berth_id": 0, "start_time": 0.0, "end_time": 3.0},
    {"vessel_id": 2, "berth_id": 1, "start_time": 1.0, "end_time": 3.0},
    {"vessel_id": 3, "berth_id": 0, "start_time": 3.0, "end_time": 7.0},
    {"vessel_id": 4, "berth_id": 1, "start_time": 3.0, "end_time": 4.0},
]
expected_total_waiting_time = 1.0
expected_makespan = 7.0
actual_by_vessel = {
    assignment["vessel_id"]: assignment
    for assignment in solution["schedule"]
}

schedule_match = all(
    actual_by_vessel[expected["vessel_id"]]["berth_id"] == expected["berth_id"]
    and math.isclose(
        actual_by_vessel[expected["vessel_id"]]["start_time"],
        expected["start_time"],
        abs_tol=1e-9,
    )
    and math.isclose(
        actual_by_vessel[expected["vessel_id"]]["end_time"],
        expected["end_time"],
        abs_tol=1e-9,
    )
    for expected in expected_schedule
)
waiting_time_match = math.isclose(
    solution["total_waiting_time"], expected_total_waiting_time, abs_tol=1e-9
)
makespan_match = math.isclose(
    solution["makespan"], expected_makespan, abs_tol=1e-9
)

print("\nReference schedule check:")
for assignment in expected_schedule:
    print(
        f"Vessel {assignment['vessel_id']} -> Berth {assignment['berth_id']}, "
        f"start={assignment['start_time']:.1f} h, end={assignment['end_time']:.1f} h"
    )
print(f"Schedule match: {schedule_match}")
print(f"Total waiting time: {solution['total_waiting_time']:.1f} hours")
print(f"Reference waiting time match: {waiting_time_match}")
print(f"Makespan: {solution['makespan']:.1f} hours")
print(f"Reference makespan match: {makespan_match}")

In [ ]:
# The Gantt chart shows berth occupancy; the second panel separates waiting cost by vessel.
print("Generating the schedule visualization.")
fcfs_formulation.visualize_schedule()

print("\nBerth utilization")
berth_utilization = {}
makespan = fcfs_formulation.solution["makespan"]

for berth in berths:
    berth_assignments = [
        assignment
        for assignment in fcfs_formulation.schedule
        if assignment["berth_id"] == berth.id
    ]
    processing_total = sum(
        assignment["processing_time"] for assignment in berth_assignments
    )
    utilization = 100.0 * processing_total / makespan if makespan else 0.0
    berth_utilization[berth.id] = utilization
    print(f"Berth {berth.id}: {utilization:.1f}%")

average_utilization = float(np.mean(list(berth_utilization.values())))
print(f"Average berth utilization: {average_utilization:.1f}%")

In [ ]:
# Vary processing times to observe how arc costs and reachable schedules respond.
print("Sensitivity analysis: processing-time scenarios")

scenarios = [
    "Base case",
    "All processing times +50%",
    "All processing times -50%",
    "Vessel 3 processing time doubled",
]
results = []

for scenario in scenarios:
    test_vessels = []
    for vessel in vessels:
        processing_time = vessel.processing_time
        if scenario == "All processing times +50%":
            processing_time *= 1.5
        elif scenario == "All processing times -50%":
            processing_time *= 0.5
        elif scenario == "Vessel 3 processing time doubled" and vessel.id == 3:
            processing_time *= 2.0

        test_vessels.append(
            Vessel(
                id=vessel.id,
                arrival_time=vessel.arrival_time,
                processing_time=processing_time,
                max_length=vessel.max_length,
                max_draft=vessel.max_draft,
            )
        )

    test_formulation = NetworkFlowFormulation(
        vessels=test_vessels,
        berths=berths,
        planning_horizon=24.0,
        time_step=0.5,
    )
    test_formulation.create_network_structure()
    test_solution = test_formulation.solve_network_flow()
    results.append(
        {
            "scenario": scenario,
            "total_waiting": test_solution["total_waiting_time"],
            "makespan": test_solution["makespan"],
            "average_waiting": test_solution["total_waiting_time"] / len(test_vessels),
        }
    )

print("\nScenario results:")
print(f"{'Scenario':<38}{'Total wait (h)':>16}{'Average (h)':>14}{'Makespan (h)':>15}")
print("-" * 83)
for result in results:
    print(
        f"{result['scenario']:<38}"
        f"{result['total_waiting']:>16.1f}"
        f"{result['average_waiting']:>14.2f}"
        f"{result['makespan']:>15.1f}"
    )

scenario_names = [result["scenario"] for result in results]
waiting_times = [result["total_waiting"] for result in results]
makespans = [result["makespan"] for result in results]
positions = np.arange(len(scenario_names))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.bar(positions, waiting_times, color="skyblue", edgecolor="black", alpha=0.85)
ax1.set_xlabel("Processing-time scenario")
ax1.set_ylabel("Total waiting time (hours)")
ax1.set_title("Total Waiting Time by Scenario")
ax1.set_xticks(positions)
ax1.set_xticklabels(scenario_names, rotation=35, ha="right")
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(positions, makespans, color="lightcoral", edgecolor="black", alpha=0.85)
ax2.set_xlabel("Processing-time scenario")
ax2.set_ylabel("Makespan (hours)")
ax2.set_title("Makespan by Scenario")
ax2.set_xticks(positions)
ax2.set_xticklabels(scenario_names, rotation=35, ha="right")
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(
    "Processing-time changes can alter both the waiting-time objective and the "
    "completion horizon. The reported values are outcomes of the same discretized "
    "network model under each scenario."
)

## Method summary

The formulation turns each feasible sequence of vessel-to-berth decisions into a path through a layered network. State information preserves the availability of every berth and the last service start, so each path respects berth capacity, vessel compatibility when dimensions are known, arrival times, and the stated FCFS rule. Waiting time is the cost of an assignment arc; minimizing path cost therefore minimizes total vessel waiting time.

The dynamic-programming calculation is exact for the specified discretized model: every feasible transition is represented, and the network is acyclic because each transition schedules the next vessel. Its state space can become large as the number of berths and time ticks grows. The result should therefore be interpreted together with the time-step assumption and model scope. Sensitivity scenarios show how changing processing requirements affects the same objective and schedule metrics without changing the method.